In [ ]:
# Summary:

# Developing a semi-supervised learning model on Diabetes dataset 
# using a super learner and deploying the model on other datasets.

# Step 1: Data pre-processing phase

# Remove outliers from all columns.
# Impute missing values in all columns.
# Normalize all columns.

# Step 2: Unsupervised Learning for generating labels

# Use K-means clustering on three features of Glucose, 
# BMI and Age to cluster data into two clusters.
# Assign ‘Diabetes’ name to the cluster with higher average Glucose 
# and ‘No Diabetes’ to the other cluster.
# Add a new column (Outcome) to the dataset containing 1 for ‘Diabetes’ 
# and 0 for ‘No Diabetes’. Use these values as labels for classification (step 4).

# Step 3: Feature Extraction

# Split data into test and training sets (consider 20% for test).
# Use PCA on the training data to create 3 new components 
# from existing features (all columns except outcome).
# Transfer training and test data to the new dimensions (PCs).

# Step 4: Classification using a super learner

# Define three classification models as base classifiers 
# consisting of Naïve Bayes, Neural Network, and KNN.
# Define a decision tree as the meta learner.
# Train decision tree (meta learner) on outputs of three base classifiers 
# using 5-fold cross validation.
# Find hyperparameters for all these models which provide the best accuracy rate.
# Report accuracy of the model on the test data.

# Step 5: Employing the model on other datasets

# Use the last column of the assigned dataset as outcome (label).
# Use your current code for steps 1,3, and 4 
# with minor changes (e.g., encoding categorical variables) 
# to train your model on the new dataset and calculate the accuracy.
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
data = pd.read_csv('diabetes_project.csv')
# print(data.head(5))
data.info()

In [ ]:
print(data.head(5))
print(data.isnull().sum())

In [ ]:
# PRE-PROCESSING
data_no_outliers = data.copy()
features = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'BMI', 'DiabetesPedigreeFunction', 'Age']
for col in features:
    Q1 = data_no_outliers[col].quantile(0.25)
    Q3 = data_no_outliers[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - (1.5 * IQR)
    upper_bound = Q3 + (1.5 * IQR)
    data_no_outliers.loc[
        (data_no_outliers[col] < lower_bound) | 
        (data_no_outliers[col] > upper_bound),
        col
    ] = np.nan
print(data_no_outliers.isnull().sum())

comlumns_to_impute = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'BMI', 'DiabetesPedigreeFunction', 'Age']
medians = data_no_outliers[comlumns_to_impute].median()
print("medians is:")
print(medians)
data_no_outliers[comlumns_to_impute] = data_no_outliers[comlumns_to_impute].fillna(medians)
print(data_no_outliers.isnull().sum())

scaler = StandardScaler()
data_scaler = scaler.fit_transform(data_no_outliers)
print(data_scaler[:5])


In [ ]:
# Step 2: Unsupervised Learning for generating labels

from sklearn.cluster import KMeans

kmeans_features = ['Glucose', 'BMI', 'Age']
data_for_kmeans = data_no_outliers[kmeans_features]
kmeans_scaler = StandardScaler()
data_kmeans_scaled = kmeans_scaler.fit_transform(data_for_kmeans)

chose_k = 2
kmeans = KMeans(n_clusters = chose_k, init = 'k-means++',n_init = 'auto', random_state = 42)
kmeans.fit(data_kmeans_scaled)
cluster_labels = kmeans.labels_
# come back to the original data to analysis
data_no_outliers['Cluster'] = cluster_labels
cluster_analysis = data_no_outliers.groupby('Cluster')[kmeans_features].mean()
print(cluster_analysis)

In [ ]:
# Assign ‘Diabetes’ name to the cluster with higher average Glucose 
# and ‘No Diabetes’ to the other cluster.
# Add a new column (Outcome) to the dataset containing 1 for ‘Diabetes’ 
# and 0 for ‘No Diabetes’. Use these values as labels for classification (step 4).
diabetes_cluster_labels = cluster_analysis['Glucose'].idxmax()
no_diabetes_cluster_labels = 1 - diabetes_cluster_labels
print(f"diabetes with high average Glucose cluster is: Cluster {diabetes_cluster_labels}")
print(f"no diabetes with high average Glucose cluster is: Cluster {no_diabetes_cluster_labels}")
# we use Map to distribute the labels
label_map = {
    diabetes_cluster_labels: 1,  # 1 = 'Diabetes'
    no_diabetes_cluster_labels: 0   # 0 = 'No Diabetes'
}
data_no_outliers['Outcome'] = data_no_outliers['Cluster'].map(label_map)
print(data_no_outliers[['Cluster','Outcome']].head(10))


In [ ]:
# Step 3: Feature Extraction

from psutil import net_connections
from sklearn.metrics import classification_report
from sklearn.decomposition import PCA


target = 'Outcome'

X = data_no_outliers[features]
y = data_no_outliers[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
PCA_scaler = StandardScaler()
X_train_scaled = PCA_scaler.fit_transform(X_train)
X_test_scaled = PCA_scaler.transform(X_test)

# Use PCA on the training data to create 3 new components from existing features (all columns except outcome).
pca = PCA(n_components = 3)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
print(f"original features: {X_train_scaled.shape}")
print(f"PCA train features: {X_train_pca.shape}")
print(f"PCA test features: {X_test_pca.shape}")


In [ ]:
# Step 4: Classification using a super learner

from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# create instance for three classification models
base_nb = GaussianNB()
base_nn = MLPClassifier(max_iter=1000, random_state=42)
base_knn = KNeighborsClassifier()
# create the meta learner
meta_learner = DecisionTreeClassifier(random_state=42)
level0_estimators = [
    ('nb', base_nb),
    ('nn', base_nn),
    ('knn', base_knn)
]
#create super learner using 5-fold cross validation
stacking_model = StackingClassifier(
    estimators=level0_estimators,
    final_estimator=meta_learner,
    cv=5
)
model_pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('superlearner', stacking_model)
])
param_grid = {
    # KNN
    'superlearner__knn__n_neighbors': [3, 5, 7, 9, 11],
    'superlearner__knn__weights': ['uniform', 'distance'],

    # Neural Network
    'superlearner__nn__hidden_layer_sizes': [(25,), (50,), (100,), (25, 25)],
    'superlearner__nn__alpha': [0.0001, 0.001, 0.01],

    # Meta Learner (Decision Tree)
    'superlearner__final_estimator__max_depth': [3, 5, 7, 10],
    'superlearner__final_estimator__min_samples_leaf': [1, 5, 10]
}
grid_search = GridSearchCV(
    estimator=model_pipeline,  # 传入 Pipeline
    param_grid=param_grid, 
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train_pca, y_train)
print(f"best hyperparameters: \n{grid_search.best_params_}")
print(f"\nbest score: {grid_search.best_score_:.4f}")

y_pred_test = grid_search.predict(X_test_pca)
final_accuracy = accuracy_score(y_test, y_pred_test)
print(f"Accuracy: {final_accuracy:.4f}")
print(classification_report(y_test, y_pred_test))

final_report = classification_report(y_test, y_pred_test)
print("\nClassification Report:")
print(final_report)